# Qwen Multimodal Chat on Colab

既定は abliterated Qwen3.8-27B（チャット・画像理解）＋ Qwen-Image-2.1（画像生成・編集）を、ブラウザからChatGPTのように使うための起動用Notebookです。

1. ［ランタイム］→［ランタイムのタイプを変更］で **A100** を選ぶ（**A100 が必須**です。L4 では現在動きません）
2. 左の🔑 **Secrets** に `HF_TOKEN` を登録（任意だけど推奨。ダウンロードが速く安定します）
   - Web検索をより安定させたい場合は `TAVILY_API_KEY`（または `BRAVE_API_KEY`）も登録。未登録でも DuckDuckGo でキー無し検索できます
3. 上から順に Cells 1〜4 を実行。表示されたリンクからチャット画面を開きます。Cell 5 は利用終了時の操作です。

画像は複数枚添付でき、PDF・短い動画の読解、マスク編集、マイク入力、読み上げ、リモートImage APIも使えます。

ロジックはすべて `src/qmc/` にあり、このNotebookは起動するだけです。詳しくは README を見てください。

In [1]:
# Cell 1: リポジトリを取得
import os
import subprocess

REPO = "https://github.com/moruku36/qwen-multimodal"
if not os.path.exists("/content/qwen-multimodal-colab"):
    subprocess.run(["git", "clone", "--depth", "1", REPO, "/content/qwen-multimodal-colab"], check=True)
else:
    subprocess.run(["git", "-C", "/content/qwen-multimodal-colab", "pull", "--ff-only"], check=True)
%cd /content/qwen-multimodal-colab
!git log --oneline -1

/content/qwen-multimodal-colab
256531d (HEAD -> main, origin/main, origin/HEAD) Merge pull request #19 from moruku36/feat/open-content-policy


In [2]:
# Cell 2: 依存関係（Colab標準のtorch/CUDAはそのまま）+ llama.cpp（2回目以降はDriveキャッシュから数秒）
%pip install -q -r requirements-colab.txt
import sys

sys.path.insert(
    0, "/content/qwen-multimodal-colab/src"
)  # editable install is not visible in a running kernel
from qmc import colab

colab.setup()  # Drive マウント（llama.cpp のキャッシュ先にも使う）
if not os.environ.get("QMC_CHAT_BASE_URL"):
    colab.install_llama_cpp()

Drive: ✅ マウント済み
履歴の保存先: /content/drive/MyDrive/qwen-multimodal-colab/data
読み込んだ Secrets: なし（HF_TOKEN 推奨）
✅ llama-server: /content/llama-bin/llama-server


PosixPath('/content/llama-bin/llama-server')

In [3]:
# Cell 3: 設定（必要なら変更）
SHARE = False  # True にすると *.gradio.live の公開URLが発行されます（誰でもアクセス可能！）
PREFETCH = False  # True: 先にモデルを全部ダウンロード（初回は大容量: Chat GGUF 約27GB + Qwen-Image-2.1 など）
PROFILE = None  # None=自動判定 / "a100_80" / "a100_40" / "l4"
IMAGE_BAND = 1024  # 標準。768=下書き / 2048=従来の高解像度（重い）
IMAGE_STEPS = 40  # 20=下書き / 50=従来の品質設定。少ないと細部・文字の品質が落ちる場合があります
os.environ["QMC_IMAGE_BAND"] = str(IMAGE_BAND)
os.environ["QMC_IMAGE_STEPS"] = str(IMAGE_STEPS)
WEB_SEARCH = "on"  # "on"=常に / "auto"=必要な質問だけ / "off"

if PREFETCH:
    colab.prefetch_models()
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv

name, memory.total [MiB], memory.used [MiB]
NVIDIA A100-SXM4-80GB, 81920 MiB, 0 MiB


In [ ]:
# Cell 4: 起動（正常に起動した UI はバックグラウンドで動き続けます）
try:
    app = colab.launch(share=SHARE, profile=PROFILE, web_search=WEB_SEARCH)
except BaseException:
    try:
        colab.shutdown_runtime()
    except Exception as cleanup_error:
        print(f"自動終了に失敗しました。［ランタイム］→［セッションを終了］: {cleanup_error}")
    raise


In [ ]:
# Cell 5: 利用終了時のセッション終了（このセルの実行だけでは終了しません）
# 生成処理の終了と必要なファイルの保存を確認してから、チェックを入れて「セッションを終了」を押してください。
from qmc import colab

colab.show_shutdown_button()


### メモ
- 初回はモデルのダウンロード（27B GGUF (Q8_K_L) 約27GB + Qwen-Image-2.1 約40GB）に時間がかかります。
- **GPU は A100（80GB 推奨）が必須**です。現在 L4 / A100 40GB は対応していません。
- 会話と画像は Google Drive の `MyDrive/qwen-multimodal-colab/data/` に保存され、再起動後も復元されます。
- 終了時は Cell 5 のチェックとボタンで、アプリ停止・Drive 同期を試みてからセッションを終了します。「すべてのセルを実行」だけでは終了しません。
- Cell 4 の起動エラー・割り込み時にも終了を試みます。Cells 1〜3 のエラー、起動後の UI エラー、カーネル停止には自動対応しません。
- `/content` のローカルファイルは終了で消えます。Drive 同期失敗・タイムアウト時は未同期データも失われる可能性があります。必要なファイルを先に保存してください。
- 自動終了できない場合やカーネルが応答しない場合は［ランタイム］→［セッションを終了］を実行してください。
- 既定では毎ターンWeb検索して出典付きで答えます（画面の 🌐 Web検索 で 自動/常に/オフ を切替）。

- 画像設定は標準1024帯・40 steps・1枚。UIの「高品質2048帯 / 50 steps」で従来設定に戻せます。下書き768帯 / 20 stepsは品質とのトレードオフです。
- 検索・プロンプト最適化の前処理時間と、ロードを含む画像処理時間を分けて表示します。CU節約率・実GPUの速度は未測定です。
- モデル解放や生成の停止だけではColabの課金対象ランタイムは終了しません。利用終了時はCell 5でセッションを終了してください。

公式 Instruct に戻す場合は、次を別のコードセルで実行してから Cell 4 を再実行してください。

```python
import os
os.environ["QMC_CHAT_HF_REPO"] = "ggml-org/Qwen3.8-27B-GGUF"
os.environ["QMC_CHAT_MODEL_FILE"] = "Qwen3.8-27B-Q4_K_M.gguf"
os.environ["QMC_CHAT_MMPROJ_REPO"] = "ggml-org/Qwen3.8-27B-GGUF"
```
